*Notebook 03 in `notebooks/arcs/`. Parameters (pre-registered): `analysis/arcs/config.yaml`; decisions and run notes: `analysis/arcs/LOG.md`. The static page `site/arcs/index.html` is built from the same saved outputs.*

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # notebooks/arcs -> repo root
sys.path.insert(0, str(REPO))

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

from analysis.arcs import s03_instruments as step

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Step 3 · Sentiment instruments

**What we do.** We score every sentence with VADER, a word lexicon with rules for negation, intensifiers,
capitals and "but". Each sentence gets a compound score in [-1, 1], and |score| < 0.05 counts as neutral. Two
contextual transformer models score the same sentences on DTU HPC:
- DistilBERT fine-tuned on SST-2, which is pre-registered;
- a three-class RoBERTa, which is robustness only.

A scene's score is the mean of its sentence scores (neutral sentences included). That is what the arcs in
Step 5 are built from.

**Why two kinds of instrument.**
- A lexicon is fast and transparent: every score can be traced to words. It does not understand order,
  sarcasm, or what an event means.
- A contextual model reads the whole sentence. It was trained on review text, not screenplays, though.

Where the two disagree, the arc depends on the instrument. Step 9 reports the analysis with each.

**What this step checks.**
1. How scores are distributed in dialogue and in action lines.
2. Whether the contraction fix from Step 2 changes any VADER score.
3. A small set of constructed stress sentences. The "intent" column is the author's construction goal, not a
   human validation; human validation is Step 4.
4. Agreement between the instruments, by rules fixed before the contextual scores were read.

No outcome is used in this step.

In [ ]:
res = step.run()   # recomputes this step and saves its outputs

**Table.** What was scored, how fast, and the state of the contextual run.

In [ ]:
display(step.tables(res)["scoring"])

,item,value
0,instrument,"VADER 3.3.2 compound score, one per sentence"
1,"sentences scored (all 1,087 scripts)","3,149,729"
2,… of which in the primary 834,"2,411,761"
3,time on one laptop core,24 s
4,neutral band,|score| < 0.05
5,contextual instruments,primary instrument by the fallback rule: sst2


**Table.** Share of negative, neutral and positive sentences per layer, primary 834 first and all 1,087 scripts second.

In [ ]:
display(step.tables(res)["distribution"])

,sample,layer,sentences,negative,neutral,positive,exactly zero,mean,median |score| if non-neutral
0,primary 834,both,2411761,17.9%,61.4%,20.6%,60.8%,0.013,0.402
1,primary 834,dialogue,1299310,16.0%,60.2%,23.8%,59.7%,0.030,0.402
2,primary 834,action,1112451,20.2%,62.8%,17.0%,62.1%,-0.007,0.402
3,"all 1,087",both,3149729,18.1%,61.5%,20.4%,60.9%,0.011,0.402
4,"all 1,087",dialogue,1675413,16.0%,60.6%,23.4%,60.1%,0.029,0.402
5,"all 1,087",action,1474316,20.5%,62.6%,17.0%,61.9%,-0.009,0.402


**Figure.** Stacked shares per layer (primary 834). Action lines are mostly neutral to a lexicon: they describe events in plain words.

In [ ]:
step.figures(res)["layers"].show()

**Figure.** Distribution of non-zero sentence scores, dialogue against action, each as a percentage of its own layer.

In [ ]:
step.figures(res)["nonzero"].show()

**Table.** Every sentence that contains a contraction, re-scored in its split ('do n't') form: how many scores move. The second row is limited to the pre-tokenised scripts, where the split form actually occurred.

In [ ]:
display(step.tables(res)["contraction"])

,set,sentences,score_changed,largest_change,crossed_neutral_band
0,every sentence with a contraction,690303,9907,1.513,4395
1,… in the 471 pre-tokenised scripts,306182,4392,1.334,2044


**Table.** Constructed stress sentences (not from any script). The contextual columns were scored on the HPC login node's CPU in fp32; the full job runs in fp16 on a GPU, so the last decimal can differ.

In [ ]:
display(step.tables(res)["stress"])

,group,sentence,constructed intent (author),vader,sst2,roberta3,roberta3_neutral,modernbert3,modernbert3_neutral
0,negation,I love you.,+,0.637,1.000,0.920,0.062,0.651,0.009
1,negation,I don't love you.,-,-0.522,-0.993,-0.729,0.226,-0.903,0.023
2,negation,This is not bad.,+ (mild),0.431,0.999,0.566,0.323,0.023,0.267
3,negation,I'm not unhappy.,+ (mild),0.325,0.998,0.202,0.528,0.474,0.017
4,negated violence,I killed him.,-,-0.670,-0.998,-0.658,0.273,0.111,0.618
5,negated violence,I didn't kill him.,0 / relief,0.577,0.987,-0.547,0.406,0.118,0.345
6,contrast and order,"The hero died, but saved everyone.","mixed, ends +",0.572,0.998,0.076,0.514,0.375,0.509
7,contrast and order,"The hero saved everyone, but died.","mixed, ends -",-0.402,-0.808,-0.600,0.320,0.092,0.765
8,word order,The dog bit the man.,-,0.000,-0.979,-0.524,0.434,-0.006,0.993
9,word order,The man bit the dog.,- (same words),0.000,-0.976,-0.509,0.453,-0.004,0.995


**Table.** The contraction forms that most often appear in sentences whose score moved, against how often they appear in all contraction sentences (single tokens only).

In [ ]:
display(step.tables(res)["contraction_forms"])

,form,in moved sentences,in all contraction sentences
0,i'm,1421,66404
1,it's,842,73408
2,don't,737,63478
3,god's,679,679
4,you're,524,39235
5,hell's,396,397
6,he's,307,36995
7,what's,260,14047
8,that's,241,32175
9,friend's,203,203


**Private check.** ten real sentences at the extremes of VADER (≥ 0.9 or ≤ −0.9), 3 to 15 words (notebook only).

In [ ]:
display(step.private(res)["extremes"])   # output cleared before commit

**Table.** Scene scores in the primary sample, per layer: quantiles and the share of scenes that score exactly zero.

In [ ]:
display(step.tables(res)["scenes"])

,layer,scenes with a score,exactly zero,q5,q25,q50,q75,q95,mean
0,both,118004,13.0%,-0.218,-0.053,0.000,0.064,0.210,0.002
1,dialogue,80256,16.2%,-0.204,-0.024,0.009,0.091,0.273,0.027
2,action,115968,20.1%,-0.262,-0.074,0.000,0.052,0.239,-0.008


**Figure.** Scene scores (both layers) in the primary sample.

In [ ]:
step.figures(res)["scenes"].show()

**Table.** Per film: the mean scene score and the SD of scene scores. That SD is a raw version of D1 before smoothing; no outcome is shown.

In [ ]:
display(step.tables(res)["films"])

,per film (primary 834),count,mean,std,min,10%,50%,90%,max
0,mean scene score per film,834.0,0.005,0.035,-0.113,-0.037,0.004,0.047,0.189
1,SD of scene scores per film,834.0,0.127,0.032,0.031,0.088,0.124,0.168,0.248


**Table.** Agreement between VADER and the contextual models, using the rules pre-registered in `instruments.agreement`.

In [ ]:
display(step.tables(res)["agreement"])

,sample,instrument,sign agreement on VADER-non-neutral sentences,"Spearman, sentences","Spearman, scenes","Spearman, films","Cohen's kappa, 3 classes",neutral share
0,primary 834,sst2,0.762,0.429,0.392,0.540,NaN,NaN
1,primary 834,roberta3,0.420,0.475,0.552,0.746,0.370,0.718
2,primary 834,modernbert3,0.313,0.341,0.421,0.676,0.192,0.699
3,"all 1,087",sst2,0.759,0.423,0.380,0.488,NaN,NaN
4,"all 1,087",roberta3,0.415,0.469,0.541,0.710,0.363,0.720
5,"all 1,087",modernbert3,0.309,0.335,0.408,0.625,0.186,0.698


**Table.** Spearman correlation between every pair of instruments that ran, primary 834: per sentence, per scene (mean of its sentences) and per film (mean of its scenes).

In [ ]:
display(step.tables(res)["matrix"])

,level,instrument,vader,sst2,roberta3,modernbert3
0,sentences,vader,1.000,0.429,0.475,0.342
1,sentences,sst2,0.429,1.000,0.656,0.503
2,sentences,roberta3,0.475,0.656,1.000,0.547
3,sentences,modernbert3,0.342,0.503,0.547,1.000
4,scenes,vader,1.000,0.392,0.552,0.421
5,scenes,sst2,0.392,1.000,0.567,0.450
6,scenes,roberta3,0.552,0.567,1.000,0.595
7,scenes,modernbert3,0.421,0.450,0.595,1.000
8,films,vader,1.000,0.540,0.746,0.676
9,films,sst2,0.540,1.000,0.685,0.661


**Figure.** Scene scores from VADER (x) against DistilBERT SST-2 (y), primary sample.

In [ ]:
step.figures(res)["agreement"].show()

**Private check.** 20 sentences where VADER and SST-2 have opposite signs, drawn by the pre-registered rule (notebook only).

In [ ]:
display(step.private(res)["disagreements"])   # output cleared before commit

**Plain-language reading** (generated from the saved numbers):

In [ ]:
display(Markdown(step.reading(res)))

VADER scored all 3,149,729 sentences in 24 seconds. In the primary sample 61% of sentences fall inside the neutral band: 60% of dialogue and 63% of action lines. Of the rest, dialogue leans positive (24% positive vs 16% negative) and action lines lean negative (20% vs 17%). Re-scoring every sentence that holds a contraction in its split, pre-tokenised form changed 9,907 of 690,303 scores (1.4%; largest change 1.51; 4,395 crossed the neutral band). The forms most often involved are "i'm", "it's", "don't", "god's". An extra token shifts VADER's three-word negation and booster windows, and a split possessive exposes a lexicon word ("God 's"). So the contraction fix is not neutral for VADER after all. What it guarantees is that pre-tokenised and ordinary scripts are scored the same way. The stress tests show the blind spots of a word lexicon: "I didn't kill him." scores +0.58; sarcasm reads as positive; violent action lines with no sentiment words score 0. VADER's "but" rule does follow order ("died, but saved everyone" +0.57, "saved everyone, but died" -0.40). On the same sentences the contextual models differ in telling ways. SST-2, which has no neutral class, gives 3 of the 3 plain action lines a score beyond ±0.5 ("He opens the door and walks in." +1.00); the three-class RoBERTa calls them mostly neutral (P(neutral) 0.81 to 0.90). "I didn't kill him." is +0.99 for SST-2 and -0.55 for RoBERTa. Both models read "Oh great, another flat tire." as negative, and both miss the sarcasm of loving to wait in line. No instrument is right on every case; this is why Step 4 validates against human labels. On the scripts themselves the instruments agree only moderately. Against VADER, SST-2 has the same sign on 76% of the sentences VADER calls non-neutral, and the Spearman correlation is 0.43 for sentences, 0.39 for scenes and 0.54 for films. The three-class RoBERTa calls 72% of sentences neutral (κ = 0.37 with VADER's three labels), and correlates 0.55 with VADER for scenes and 0.75 for films. By the pre-registered fallback rule (projected 0.14 h ≤ 3 h) the primary instrument is DistilBERT SST-2 (sst2); VADER and RoBERTa are robustness instruments (Step 9). The ModernBERT-large model added later (amendment (d), robustness only) calls 70% of sentences neutral (κ = 0.19 with VADER). At film level it correlates 0.66 with SST-2, 0.76 with RoBERTa and 0.68 with VADER.